# 03. Cortex Search ― PDF レポートを検索できるようにする

02 の Cortex Analyst は「テーブルの数字」に答えます。一方、**「推奨カバレッジは？」「停滞商談にはどう対応すべき？」** のような答えは PDF のレポートにあります。
**Cortex Search** は、文書を意味で検索できるようにするサービスです。

```
PDF（DOCS_STAGE。中身は GitHub の data/pdfs/ で見られます）
  ↓ ① AI_PARSE_DOCUMENT   … PDF からテキストを抽出（表は Markdown に）
  ↓ ② SPLIT_TEXT_RECURSIVE_CHARACTER … 検索しやすい長さ（チャンク）に分割
  ↓ ③ CREATE CORTEX SEARCH SERVICE  … ベクトル＋キーワードのハイブリッド検索インデックス
Cortex Search Service `SALES_DOCS_SEARCH` → 04 で Agent のツールにする
```

**前提**：`02_cortex_analyst.ipynb` を実行済みであること

## Step 0. セッションの準備

Notebook はそれぞれ別の Python 実行環境で動くため、前の Notebook で作った変数や接続は引き継がれません。
そこで各 Notebook の最初に、Snowflake への接続（`session`）を取得し、ロール・ウェアハウス・スキーマをそろえます。
Python のセルから SQL を実行するには、この `session` が必要です。

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()   # Notebook がすでに張っている Snowflake への接続を取得

DB, SCHEMA = "CORTEX_QUICK_HOL_DB", "CORTEX_QUICK_HOL_SCHEMA"
HOL_ROLE, HOL_WH = "CORTEX_QUICK_HOL_ROLE", "CORTEX_QUICK_HOL_WH"

session.use_role("ACCOUNTADMIN")
session.use_warehouse(HOL_WH)
session.use_schema(f"{DB}.{SCHEMA}")
print("ロール:", session.get_current_role(), "/ ウェアハウス:", session.get_current_warehouse())

## Step 1. PDF からテキストを取り出す

`AI_PARSE_DOCUMENT` の `LAYOUT` モードは、見出しや**表の構造を保ったまま** Markdown のテキストにします。

In [ ]:
%%sql -r parse_docs
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_PARSED AS
SELECT RELATIVE_PATH AS FILE_NAME,
       AI_PARSE_DOCUMENT(TO_FILE('@CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_STAGE', RELATIVE_PATH), {'mode': 'LAYOUT'}):content::STRING AS CONTENT
FROM DIRECTORY(@CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_STAGE)

In [ ]:
%%sql -r show_parsed
SELECT FILE_NAME, LENGTH(CONTENT) AS CHARS, LEFT(CONTENT, 500) AS HEAD FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_PARSED

## Step 2. チャンクに分割する

PDF 1 本を丸ごと 1 件として登録せず、短い「チャンク」に分けてから検索サービスに入れます。理由は 3 つです。

| 理由 | 内容 |
|---|---|
| ベクトル化できる長さに上限がある | 一度に読める長さは埋め込みモデルによって違い、今回使う `snowflake-arctic-embed-l-v2.0` は **512 トークン**です（`-8k` 版は 8,192、`voyage-multilingual-2` は 32,000）。上限を超えた部分は**ベクトル検索では使われません**（キーワード検索では全文が使われます）。長く読めるモデルでも、Snowflake は 1 チャンク 512 トークン以下を推奨しています |
| 検索の精度が上がる | 1 件の中に話題がたくさん混ざると、質問との「意味の近さ」がぼやけます。話題ごとに分けると、関係する段落だけが上位に来ます |
| Agent に渡す量が適切になる | Agent は検索結果を読んで回答を作ります。PDF 全体ではなく関係する段落だけを渡すほうが、回答が正確で速くなります |

ここでは約 800 文字ずつ（前後 100 文字は重ねて、文の途中で切れても前後の文脈が残るように）分割します。
`'markdown'` を指定すると、見出しや段落の切れ目を優先して分割します。

In [ ]:
%%sql -r chunk_docs
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_CHUNKS AS
SELECT d.FILE_NAME || '#' || c.INDEX AS CHUNK_ID,
       d.FILE_NAME,
       c.INDEX        AS CHUNK_NO,
       c.VALUE::STRING AS CHUNK
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_PARSED d,
     LATERAL FLATTEN(SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(d.CONTENT, 'markdown', 800, 100)) c

In [ ]:
%%sql -r chunk_counts
SELECT FILE_NAME, COUNT(*) AS CHUNKS, ROUND(AVG(LENGTH(CHUNK))) AS AVG_CHARS
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_CHUNKS GROUP BY FILE_NAME

## Step 3. Cortex Search Service を作る

| 設定 | 値 | 意味 |
|---|---|---|
| `ON CHUNK` | 検索対象の列 | この列の中身で検索する |
| `ATTRIBUTES FILE_NAME` | 絞り込みに使える列 | 「このレポートだけ」で絞り込める |
| `EMBEDDING_MODEL` | `snowflake-arctic-embed-l-v2.0` | **多言語対応**のモデル。英語の PDF を日本語の質問で検索できる |
| `TARGET_LAG` | `1 day` | 元のテーブルが変わったときに追従する間隔 |

> 💡 既定のモデルは英語向けです。日本語で質問する場合は多言語モデルを指定してください（このハンズオンで実際に、既定モデルでは日本語の質問に対して関係の薄い表が上位に来ました）。
>
> 作成には 1〜2 分かかります。

### 方法 A：GUI（AI と ML → Cortex Search）で作る

Step 1・2 で作った `DOCS_CHUNKS` テーブルを、画面から検索サービスにします。

1. Snowsight 左メニュー **「AI と ML」→「Cortex Search」** を開き、**「Create」**
2. ロールは `ACCOUNTADMIN`、ウェアハウスは **`CORTEX_QUICK_HOL_WH`**
3. データベース / スキーマに `CORTEX_QUICK_HOL_DB` / `CORTEX_QUICK_HOL_SCHEMA`、サービス名に **`SALES_DOCS_SEARCH`**（**大文字のまま**）を入力し、**Next**
4. データの種類は **Table or view** → **`DOCS_CHUNKS`** を選び、**Next**
5. 検索結果に含める列：`CHUNK_ID` / `CHUNK` / `FILE_NAME` / `CHUNK_NO` → **Next**
6. 検索する列：**`CHUNK`** → **Next**
7. 絞り込みに使う列：**`FILE_NAME`** → **Next**
8. Target lag は **1 day** にして **Create**
9. 作成されたら同じ画面の **Playground** で「推奨されるパイプラインカバレッジの目安は？」と検索してみる

> ⚠️ **埋め込みモデル**：画面でモデルを選べない場合、既定の英語向けモデルで作られ、日本語の質問の精度が下がります。
> その場合は **方法 B のセルを実行**して、多言語モデル（`snowflake-arctic-embed-l-v2.0`）で作り直してください。
>
> ⚠️ GUI ではサービス名がダブルクォート付きで作られます。小文字が混じると SQL や Agent から `SALES_DOCS_SEARCH` で参照できなくなるため、**必ず大文字で入力**してください。

> ℹ️ GUI の手順は Snowflake 公式ドキュメントをもとに書いています。画面のラベルや順番が多少違う場合があります。
> 迷ったら **方法 B（SQL）** のセルを実行すれば、同じ名前・同じ設定で作られます（GUI で作ったものは上書きされます）。

### 方法 B：SQL で作る（`CREATE CORTEX SEARCH SERVICE`）

In [ ]:
%%sql -r create_search_service
CREATE OR REPLACE CORTEX SEARCH SERVICE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH
  ON CHUNK
  ATTRIBUTES FILE_NAME
  WAREHOUSE = CORTEX_QUICK_HOL_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS SELECT CHUNK_ID, CHUNK, FILE_NAME, CHUNK_NO FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_CHUNKS

GUI・SQL どちらで作った場合も、次のセルで権限を付けます。

In [ ]:
session.sql(f"GRANT USAGE ON CORTEX SEARCH SERVICE {DB}.{SCHEMA}.SALES_DOCS_SEARCH TO ROLE {HOL_ROLE}").collect()
print("OK: Cortex Search Service への USAGE を付与しました")

## Step 4. 検索してみる

`SEARCH_PREVIEW` で、質問に近いチャンクを上位から取り出します。英語の PDF に**日本語で**質問しています。

In [ ]:
def search_docs(query: str, limit: int = 3):
    req = json.dumps({"query": query, "columns": ["CHUNK", "FILE_NAME"], "limit": limit})
    res = json.loads(session.sql("SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(?, ?) AS R",
                                 params=[f"{DB}.{SCHEMA}.SALES_DOCS_SEARCH", req]).collect()[0]["R"])
    print(f"Q: {query}\n")
    for i, r in enumerate(res["results"], 1):
        print(f"[{i}] {r['FILE_NAME']}\n{' '.join(r['CHUNK'].split())[:300]}...\n")

import json
search_docs("推奨されるパイプラインカバレッジの目安は？")

In [ ]:
search_docs("停滞している商談にはどう対応すべき？")

In [ ]:
search_docs("解約リスクが高い顧客に共通する特徴は？")

> ✅ それぞれの質問に、関係する段落（推奨 1.6 倍、役員スポンサーの関与、技術的なエスカレーションなど）が上位に返れば成功です。

✅ 03 はここまでです。次は `04_cortex_agent.ipynb` で、Analyst と Search を 1 つの Agent にまとめます。